# 01 – Clip LAS to raster tiles (Site 11)

This notebook mirrors the `clipping_toRaster.py` workflow using the packaged `rasterclip` functions.

- Pre-crops in LAS/native CRS using raster bounds transformed to LAS CRS
- Reprojects output to raster CRS
- Batch Dask mode shows progress and writes a signature file

In [4]:
from pathlib import Path

# --- CONFIGURATION: Site 11 (DD0011) ---
SITE_NAME = 'site2_DD0011'

# Input LAS (choose appropriate level/coreg version)
# You can also try L1 or non-coreg versions if needed
INPUT_LAS = Path('/data/calperumResearch/site2_1_DD0011/lidar/20220517_SASMDD0011_AusGeoid20_coreg.las')


# Raster tiles directory and pattern
AOI_RASTER_DIR = Path('/data/calperumResearch/site2_1_DD0011/imagery/inputs/predictors/tiles_3072/stacked/')
RASTER_PATTERN = 'tiles_multispectral*.tif'

# Single tile example
TILE_ID = '33'
A_TILE = AOI_RASTER_DIR / f'tiles_multispectral.{TILE_ID}.tif'

# Output directory
OUT_DIR = Path('/data/calperumResearch/site2_1_DD0011/lidar/clipped/raster/coreg')
OUT_DIR.mkdir(parents=True, exist_ok=True)  # Commented out - create /data/ manually with proper permissions

print('--- Configuration ---')
print(f'LAS         : {INPUT_LAS}')
print(f'Raster dir  : {AOI_RASTER_DIR}')
print(f'Pattern     : {RASTER_PATTERN}')
print(f'Single tile : {A_TILE}')
print(f'Output dir  : {OUT_DIR}')
print('----------------------')

--- Configuration ---
LAS         : /data/calperumResearch/site2_1_DD0011/lidar/20220517_SASMDD0011_AusGeoid20_coreg.las
Raster dir  : /data/calperumResearch/site2_1_DD0011/imagery/inputs/predictors/tiles_3072/stacked
Pattern     : tiles_multispectral*.tif
Single tile : /data/calperumResearch/site2_1_DD0011/imagery/inputs/predictors/tiles_3072/stacked/tiles_multispectral.33.tif
Output dir  : /data/calperumResearch/site2_1_DD0011/lidar/clipped/raster/coreg
----------------------


In [5]:
# --- Single tile clip (optional; can be slow) ---
import time
from rasterclip import clip_single

out_single = OUT_DIR / f'{A_TILE.stem}.las'
print('Starting single-tile clip...')
t0 = time.time()

pts = clip_single(INPUT_LAS, A_TILE, out_single)
dt = time.time() - t0
print(f'✅ Single tile written: {out_single} ({pts} points) in {dt:.2f}s')
print('Tip: If single-tile is slow, run Dask batch directly for better throughput.')

Starting single-tile clip...


PDAL: Unable to open stream for '/data/calperumResearch/site2_1_DD0011/lidar/20220517_SASMDD0011_AusGeoid20_coreg.las' with error 'No such file or directory'



CalledProcessError: Command '['pdal', 'info', '--metadata', '/data/calperumResearch/site2_1_DD0011/lidar/20220517_SASMDD0011_AusGeoid20_coreg.las']' returned non-zero exit status 1.

In [21]:
# --- Batch clip: sequential and Dask with signature ---
from rasterclip import clip_many, clip_many_dask

# Sequential
outs_seq = clip_many(INPUT_LAS, AOI_RASTER_DIR, OUT_DIR, pattern=RASTER_PATTERN, skip_existing=True)
print(f'✅ Sequential batch outputs: {len(outs_seq)}')

# Dask-parallel with signature logging
outs_dask = clip_many_dask(
    INPUT_LAS,
    AOI_RASTER_DIR,
    OUT_DIR,
    pattern=RASTER_PATTERN,
    skip_existing=True,
    write_signature=True,
 )
print(f'✅ Dask batch outputs: {len(outs_dask)}')

sig_path = OUT_DIR / 'clip_signature.txt'
if sig_path.exists():
    print(f'📝 Signature file: {sig_path}')
    try:
        print(sig_path.read_text())
    except Exception:
        print('Signature written, but could not display contents.')

PermissionError: [Errno 13] Permission denied: '/data'

In [27]:
# --- Inspect one output LAS ---
import laspy, numpy as np

sample_out = out_single if out_single.exists() else (OUT_DIR / f'tiles_multispectral.{TILE_ID}.las')
print(f'Inspecting: {sample_out}')
las = laspy.read(sample_out)
print(f'Total points: {len(las.points):,}')
print('Bounding box:')
print(f'  X: {las.header.mins[0]:.2f} → {las.header.maxs[0]:.2f}')
print(f'  Y: {las.header.mins[1]:.2f} → {las.header.maxs[1]:.2f}')
print(f'  Z: {las.header.mins[2]:.2f} → {las.header.maxs[2]:.2f}')
print(f'Point format: {las.header.point_format}')
print(f'Extra dims : {[dim.name for dim in las.point_format.extra_dimensions]}')

Inspecting: /data/calperumResearch/site2_1_DD0011/lidar/clipped/raster/coreg/tiles_multispectral.33.las


FileNotFoundError: [Errno 2] No such file or directory: '/data/calperumResearch/site2_1_DD0011/lidar/clipped/raster/coreg/tiles_multispectral.33.las'

In [ ]:
# Tip: Dask dashboard
print('If running Dask in this notebook, check the dashboard in logs or via Client().dashboard_link')

In [ ]:
# --- Visualize clipped LAS over raster ---
import rasterio
from rasterio.plot import show as rio_show
import laspy
import matplotlib.pyplot as plt
import numpy as np

# Choose a tile and corresponding clipped LAS
tile_path = A_TILE
las_path = OUT_DIR / f"{A_TILE.stem}.las"
if not las_path.exists():
    # fallback to the inspected sample
    las_path = sample_out

print(f"Raster: {tile_path}")
print(f"Clipped LAS: {las_path}")

# Read raster (first band for backdrop)
src = rasterio.open(tile_path)
img = src.read(1)
extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)

# Read LAS (already reprojected to raster CRS by pipeline)
las = laspy.read(las_path)
x = las.x
y = las.y

# Downsample for plotting speed if huge
n = len(x)
max_pts = 200000
if n > max_pts:
    idx = np.random.choice(n, size=max_pts, replace=False)
    x = x[idx]
    y = y[idx]

plt.figure(figsize=(8, 8))
rio_show(img, extent=extent, cmap='gray', title='Clipped LAS over Raster')
plt.scatter(x, y, s=0.2, c='cyan', alpha=0.5)
plt.xlabel('X')
plt.ylabel('Y')
plt.tight_layout()
plt.show()

In [ ]:
# --- Visualize raster RGB (bands 5/3/1) with colored LAS points ---
import rasterio
import numpy as np
import laspy
import matplotlib.pyplot as plt

tile_path = A_TILE
las_path = OUT_DIR / f"{A_TILE.stem}.las"
if not las_path.exists():
    las_path = sample_out

print(f"Raster: {tile_path}")
print(f"Clipped LAS: {las_path}")

with rasterio.open(tile_path) as src:
    # Read bands 5,3,1 (1-indexed)
    if src.count < 5:
        raise RuntimeError(f"Raster has {src.count} bands; need >= 5 for 5/3/1 RGB.")
    b5 = src.read(5).astype(np.float32)
    b3 = src.read(3).astype(np.float32)
    b1 = src.read(1).astype(np.float32)
    def norm_band(b):
        lo, hi = np.percentile(b[np.isfinite(b)], [2, 98])
        b = np.clip((b - lo) / (hi - lo + 1e-6), 0, 1)
        b[~np.isfinite(b)] = 0
        return b
    rgb = np.stack([norm_band(b5), norm_band(b3), norm_band(b1)], axis=-1)
    extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)

las = laspy.read(las_path)
x = las.x
y = las.y

use_rgb = hasattr(las, 'red') and hasattr(las, 'green') and hasattr(las, 'blue')
if use_rgb:
    r = np.asarray(las.red, dtype=np.float32)
    g = np.asarray(las.green, dtype=np.float32)
    b = np.asarray(las.blue, dtype=np.float32)
    las_colors = np.stack([r, g, b], axis=-1) / 65535.0
else:
    z = np.asarray(las.z, dtype=np.float32)
    z_lo, z_hi = np.percentile(z, [2, 98])
    z_norm = np.clip((z - z_lo) / (z_hi - z_lo + 1e-6), 0, 1)
    las_colors = None

n = len(x)
max_pts = 200000
if n > max_pts:
    idx = np.random.choice(n, size=max_pts, replace=False)
    x = x[idx]
    y = y[idx]
    if use_rgb:
        las_colors = las_colors[idx]
    else:
        z_norm = z_norm[idx]

plt.figure(figsize=(9, 9))
plt.imshow(rgb, extent=extent, origin='upper')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Raster RGB (5/3/1) with LAS overlay')
if use_rgb:
    plt.scatter(x, y, s=0.2, c=las_colors, alpha=0.7)
else:
    plt.scatter(x, y, s=0.2, c=z_norm, cmap='viridis', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# --- View original raster (bands 5/3/1) ---
import rasterio, numpy as np, matplotlib.pyplot as plt

tile_path = A_TILE
with rasterio.open(tile_path) as src:
    if src.count < 5:
        raise RuntimeError(f"Raster has {src.count} bands; need >= 5 for 5/3/1 RGB.")
    b5 = src.read(5).astype(np.float32)
    b3 = src.read(3).astype(np.float32)
    b1 = src.read(1).astype(np.float32)
    def norm_band(b):
        lo, hi = np.percentile(b[np.isfinite(b)], [2, 98])
        b = np.clip((b - lo) / (hi - lo + 1e-6), 0, 1)
        b[~np.isfinite(b)] = 0
        return b
    rgb = np.stack([norm_band(b5), norm_band(b3), norm_band(b1)], axis=-1)
    extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)

plt.figure(figsize=(8, 8))
plt.imshow(rgb, extent=extent, origin='upper')
plt.title('Raster RGB (bands 5/3/1)')
plt.xlabel('X'); plt.ylabel('Y')
plt.tight_layout(); plt.show()

In [ ]:
# --- View LAS only (colored by RGB if available; else by Z) ---
import laspy, numpy as np, matplotlib.pyplot as plt

las_path = OUT_DIR / f"{A_TILE.stem}.las"
if not las_path.exists():
    las_path = sample_out
las = laspy.read(las_path)

x = las.x
y = las.y
use_rgb = hasattr(las, 'red') and hasattr(las, 'green') and hasattr(las, 'blue')
if use_rgb:
    r = np.asarray(las.red, dtype=np.float32)
    g = np.asarray(las.green, dtype=np.float32)
    b = np.asarray(las.blue, dtype=np.float32)
    colors = np.stack([r, g, b], axis=-1) / 65535.0
else:
    z = np.asarray(las.z, dtype=np.float32)
    z_lo, z_hi = np.percentile(z, [2, 98])
    colors = np.clip((z - z_lo) / (z_hi - z_lo + 1e-6), 0, 1)

n = len(x); max_pts = 200000
if n > max_pts:
    idx = np.random.choice(n, size=max_pts, replace=False)
    x = x[idx]; y = y[idx]
    colors = colors[idx]

plt.figure(figsize=(8, 8))
plt.title('LAS points only')
plt.xlabel('X'); plt.ylabel('Y')
if use_rgb:
    plt.scatter(x, y, s=2, c=colors, alpha=0.8)
else:
    plt.scatter(x, y, s=2, c=colors, cmap='viridis', alpha=0.8)
plt.tight_layout(); plt.show()

In [ ]:
# --- Overlay: Raster 5/3/1 + LAS ---
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 9))
plt.imshow(rgb, extent=extent, origin='upper')
plt.xlabel('X'); plt.ylabel('Y')
plt.title('Overlay: Raster 5/3/1 + LAS points')
# Reuse colors from previous cell if available
try:
    use_rgb_present = 'las_colors' in globals() or 'colors' in globals()
except Exception:
    use_rgb_present = False
if use_rgb_present:
    cvals = colors if 'colors' in globals() else las_colors
    if cvals.ndim == 2:
        plt.scatter(x, y, s=2, c=cvals, alpha=0.7)
    else:
        plt.scatter(x, y, s=2, c=cvals, cmap='viridis', alpha=0.7)
else:
    # Fallback: recompute simple Z colors
    import numpy as np, laspy
    las = laspy.read(OUT_DIR / f"{A_TILE.stem}.las")
    x = las.x; y = las.y
    z = np.asarray(las.z, dtype=np.float32)
    z_lo, z_hi = np.percentile(z, [2, 98])
    z_norm = np.clip((z - z_lo) / (z_hi - z_lo + 1e-6), 0, 1)
    plt.scatter(x, y, s=2, c=z_norm, cmap='viridis', alpha=0.7)
plt.tight_layout(); plt.show()